# SAMet — Week 10–11 Deliverable: Working Notebook with Baseline Output

**Course:** YZV416E Computer Vision, Istanbul Technical University  
**Team:** SAMet | Alper Düzgün · Enis Furkan Kırmızı · Akın Yağbasan  
**Topic:** SAM 2 Video Object Segmentation on DAVIS 2017  

---

This notebook covers:
1. Environment verification
2. SAM 2 model loading
3. DAVIS 2017 dataset loading
4. First forward pass on a single sequence
5. Visualisation of predictions vs ground truth
6. Quick J&F score on the sample sequence

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

print('Python  :', sys.version.split()[0])
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available(), '—', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')
print('Device  :', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. Configuration

In [ ]:
import yaml

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

DAVIS_ROOT  = cfg['data']['davis_root']
DEVICE      = cfg.get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SIZE = cfg['data']['resolution']   # 1024

print('DAVIS root :', DAVIS_ROOT)
print('Device     :', DEVICE)
print('Input size :', TARGET_SIZE)

## 2. Download data & checkpoint (skip if already done)

In [ ]:
# Run this cell only once.
# It downloads DAVIS 2017 (480p, ~6 GB) and the SAM 2 Hiera-Base+ checkpoint.

davis_split_file = Path(DAVIS_ROOT) / 'ImageSets' / '2017' / 'val.txt'
checkpoint_path  = Path(cfg['model']['checkpoint'])

if not davis_split_file.exists() or not checkpoint_path.exists():
    print('Downloading DAVIS 2017 and SAM 2 checkpoint...')
    os.system('python ../scripts/download_davis.py')
else:
    print('Data and checkpoint already present — skipping download.')

## 3. Load the DAVIS dataset

In [ ]:
from src.dataset import DAVISDataset, DAVISSequence

val_dataset = DAVISDataset(DAVIS_ROOT, split='val')
print(f'Validation sequences: {len(val_dataset)}')
for seq in val_dataset.sequences[:5]:
    print(f'  {seq.name:30s} — {len(seq)} frames, objects: {seq.object_ids}')

In [ ]:
# Inspect one sequence — use bmx-trees as in the proposal example
SEQ_NAME = 'bmx-trees'
try:
    seq = next(s for s in val_dataset.sequences if s.name == SEQ_NAME)
except StopIteration:
    seq = val_dataset.sequences[0]
    print(f'Sequence "{SEQ_NAME}" not found; using "{seq.name}" instead.')

frame0 = seq.load_frame(0)
mask0  = seq.load_mask(0)
prompts = seq.get_first_frame_prompts()

print(f'Sequence  : {seq.name}')
print(f'Frames    : {len(seq)}')
print(f'Objects   : {seq.object_ids}')
print(f'Frame size: {frame0.shape}')

In [ ]:
# Visualise frame 0 + ground-truth mask
from src.visualize import overlay_mask_on_frame

gt_masks_frame0 = {oid: (mask0 == oid).astype(np.uint8) for oid in seq.object_ids}
overlay = overlay_mask_on_frame(frame0, gt_masks_frame0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(frame0)
axes[0].set_title('Frame 0 — Original')
axes[0].axis('off')
axes[1].imshow(overlay)
axes[1].set_title('Frame 0 — Ground-Truth Mask')
axes[1].axis('off')
plt.suptitle(f'Sequence: {seq.name}', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Preprocessing pipeline check

In [ ]:
from src.preprocessing import preprocess_frame, resize_mask, mask_to_box, mask_to_point, encode_mask_rle, decode_mask_rle

proc_frame = preprocess_frame(frame0, size=TARGET_SIZE)
proc_mask  = resize_mask(gt_masks_frame0[seq.object_ids[0]], size=TARGET_SIZE)

print(f'Preprocessed frame shape : {proc_frame.shape}  dtype: {proc_frame.dtype}')
print(f'Preprocessed frame range : [{proc_frame.min():.2f}, {proc_frame.max():.2f}]')
print(f'Preprocessed mask shape  : {proc_mask.shape}  unique: {np.unique(proc_mask)}')

# Derived prompts
box    = mask_to_box(proc_mask)
points, labels = mask_to_point(proc_mask)
print(f'Bounding box    : {box}')
print(f'Centroid point  : {points}  label: {labels}')

# RLE round-trip
rle     = encode_mask_rle(proc_mask)
decoded = decode_mask_rle(rle)
print(f'RLE round-trip OK: {np.array_equal(proc_mask, decoded)}')

## 5. Augmentation pipeline check

In [ ]:
from src.augmentation import build_train_augmentation

aug = build_train_augmentation(cfg)
frames_in  = [seq.load_frame(i) for i in range(min(5, len(seq)))]
masks_in   = [seq.load_mask(i)  for i in range(min(5, len(seq)))]

frames_out, masks_out = aug(frames_in, masks_in)

fig, axes = plt.subplots(2, 5, figsize=(20, 6))
for i in range(5):
    axes[0, i].imshow(frames_in[i])
    axes[0, i].set_title(f'Original {i}')
    axes[0, i].axis('off')
    axes[1, i].imshow(frames_out[i])
    axes[1, i].set_title(f'Augmented {i}')
    axes[1, i].axis('off')
plt.suptitle('Augmentation: before vs after (first 5 frames)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Load SAM 2 and run first forward pass

In [ ]:
from src.inference import load_sam2_video_predictor, run_inference_on_sequence

predictor = load_sam2_video_predictor(cfg)
print('SAM 2 predictor loaded.')

In [ ]:
%%time
# First forward pass — uses GT mask from frame 0 as prompt (semi-supervised setting)
pred_masks = run_inference_on_sequence(predictor, seq, prompt_type='mask')

for oid, masks in pred_masks.items():
    print(f'Object {oid}: {len(masks)} predicted masks, '
          f'first mask shape={masks[0].shape}, '
          f'coverage={masks[0].mean()*100:.1f}%')

## 7. Visualise predictions across selected frames

In [ ]:
VIZ_FRAMES = [0, len(seq)//4, len(seq)//2, 3*len(seq)//4, len(seq)-1]

fig, axes = plt.subplots(2, len(VIZ_FRAMES), figsize=(5*len(VIZ_FRAMES), 8))

for col, fidx in enumerate(VIZ_FRAMES):
    frame  = seq.load_frame(fidx)
    gt_m   = {oid: seq.load_binary_mask(fidx, oid) for oid in seq.object_ids}
    pred_m = {oid: pred_masks[oid][fidx] for oid in pred_masks if fidx < len(pred_masks[oid])}

    gt_overlay   = overlay_mask_on_frame(frame, gt_m)
    pred_overlay = overlay_mask_on_frame(frame, pred_m)

    axes[0, col].imshow(gt_overlay)
    axes[0, col].set_title(f'GT — frame {fidx}')
    axes[0, col].axis('off')

    axes[1, col].imshow(pred_overlay)
    axes[1, col].set_title(f'SAM 2 — frame {fidx}')
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Ground Truth', fontsize=11)
axes[1, 0].set_ylabel('SAM 2 Prediction', fontsize=11)
plt.suptitle(f'Sequence: {seq.name}  |  Ground Truth vs SAM 2 Zero-Shot', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Quick J&F evaluation on this sequence

In [ ]:
from src.evaluate import evaluate_sequence

for oid in seq.object_ids:
    gt_seq   = [seq.load_binary_mask(i, oid).astype(np.uint8) for i in range(len(seq))]
    pred_seq = pred_masks.get(oid, [])

    if not pred_seq:
        print(f'Object {oid}: no predictions')
        continue

    # Align lengths
    n = min(len(gt_seq), len(pred_seq))
    result = evaluate_sequence(pred_seq[:n], gt_seq[:n], compute_f=True)
    print(f'Object {oid}:  J = {result["J"]:.4f}   F = {result["F"]:.4f}   J&F = {result["JF"]:.4f}')

## 9. Error heatmap on worst frame

In [ ]:
from src.evaluate import jaccard
from src.visualize import error_heatmap

oid = seq.object_ids[0]
j_scores = [
    jaccard(pred_masks[oid][i], seq.load_binary_mask(i, oid))
    for i in range(min(len(pred_masks[oid]), len(seq)))
]
worst_frame = int(np.argmin(j_scores))
print(f'Worst frame for object {oid}: frame {worst_frame}  (J={j_scores[worst_frame]:.4f})')

frame_w  = seq.load_frame(worst_frame)
pred_w   = pred_masks[oid][worst_frame].astype(bool)
gt_w     = seq.load_binary_mask(worst_frame, oid).astype(bool)
heatmap  = error_heatmap(pred_w, gt_w)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(frame_w)
axes[0].set_title(f'Frame {worst_frame}')
axes[0].axis('off')
axes[1].imshow(overlay_mask_on_frame(frame_w, {oid: pred_w.astype(np.uint8)}))
axes[1].set_title(f'SAM 2 Prediction  (J={j_scores[worst_frame]:.3f})')
axes[1].axis('off')
axes[2].imshow(heatmap)
axes[2].set_title('Error Map')
axes[2].axis('off')
patches = [
    mpatches.Patch(color=(0,200/255,0), label='TP'),
    mpatches.Patch(color=(200/255,0,0), label='FP'),
    mpatches.Patch(color=(0,0,200/255), label='FN'),
]
axes[2].legend(handles=patches, loc='lower right', fontsize=9)
plt.suptitle(f'Error Analysis — {seq.name} frame {worst_frame}', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Summary

| Step | Status |
|------|--------|
| Environment (PyTorch, CUDA, SAM 2) | ✅ |
| DAVIS 2017 dataset loaded | ✅ |
| Preprocessing (resize, normalize, RLE) | ✅ |
| Augmentation pipeline | ✅ |
| SAM 2 model loaded | ✅ |
| First forward pass (zero-shot) | ✅ |
| Prediction visualization | ✅ |
| J&F score (sample sequence) | ✅ |

**Next step (Weeks 11–12):** Run zero-shot evaluation on the full DAVIS 2017 validation set using `scripts/run_zero_shot.py` and record baseline J&F scores.